# **1 - Definição do Problema**

## Contexto

O naufrágio do Titanic é um dos desastres marítimos mais conhecidos da história. O conjunto de dados utilizado contém informações sobre os passageiros, incluindo características como gênero, idade, classe social, valor da tarifa e situação de sobrevivência.

## Problema

Embora o conjunto de dados contenha diversas informações sobre os passageiros, não está claro quais características estiveram mais associadas às chances de sobrevivência durante o desastre.

## Pergunta Principal

Quais características dos passageiros apresentaram maior associação com a sobrevivência no Titanic?

## Objetivo

Investigar a relação entre diferentes características dos passageiros e suas taxas de sobrevivência, utilizando técnicas de Análise Exploratória de Dados (EDA).

Para responder à pergunta principal, serão analisados fatores como:

* Gênero;
* Classe do passageiro;
* Faixa etária;
* Tamanho da família;
* Valor da tarifa.

As análises serão conduzidas por meio da comparação de métricas e taxas de sobrevivência entre diferentes grupos de passageiros.


---

# **2 - Importação**

In [ ]:
import pandas as pd
import seaborn as sns
import sys
from pathlib import Path

In [ ]:
# Configuração para poder importar os módulos
sys.path.append(str(Path("../").resolve()))

In [ ]:
from src import statistics
from src.cleaning import limpar_dados
from src.features import engenharia_features
from src.visualization import grafico_barras

---

# **3 - Leitura do Arquivo**

In [ ]:
# Cria uma variável para o caminho e lê o arquvio csv
DATA_PATH = Path("../data/raw/train.csv")
df = pd.read_csv(DATA_PATH)

---

# **4 - Exploração Inicial**

In [ ]:
# Mostrar as 5 primeiras linhas do dataset
df.head()

In [ ]:
# Cria uma descrição estatísta sobre as variáveis do dataset
df.describe(include="all")

In [ ]:
# Mostra informações sobre o dataset
df.info(memory_usage="deep")

In [ ]:
# Calcula quantos valores nulos tem em cada coluna
df.isnull().sum()

In [ ]:
# Calcula quantos valores duplicados existem no dataset
df.duplicated().sum()

### Resumo da exploração inicial

- 891 registros
- 12 variáveis
- `Cabin` possui elevado número de valores ausentes
- `Age` possui valores ausentes
- `Ticket` possui alta cardinalidade

---

# **5 - Limpeza e Tratamento**

In [ ]:
df_limpo = limpar_dados(df)

### Limpeza dos Dados

Nesta etapa foram realizadas as seguintes transformações por meio da função `limpar_dados`:

- Cria uma cópia do DataFrame e retorna ela tratada.
- Preenchimento dos valores ausentes da coluna `Age` utilizando a mediana calculada para cada grupo de `Sex` e `Pclass`, preservando melhor as características dos passageiros.
- Preenchimento dos valores ausentes da coluna `Embarked` utilizando a moda, por se tratar de uma variável categórica.
- Remoção da coluna `Cabin`, devido à alta proporção de valores ausentes, o que comprometeria sua utilização na análise.
- Remoção da coluna `Ticket`, por não agregar informações relevantes aos objetivos desta análise.

Essas transformações tiveram como objetivo tratar valores ausentes e remover atributos com baixa utilidade analítica, preservando a qualidade e a consistência da base de dados para as etapas seguintes.

In [ ]:
# Verifica se existe algum valor nulo
df_limpo.isnull().sum()

# Cria um arquivo csv com o DataFrame limpo
df_limpo.to_csv(
    "../data/processed/train_limpo.csv",
    index=False
)

In [ ]:
df_limpo = engenharia_features(df_limpo)

## Engenharia de Features

Nesta etapa foram criadas novas variáveis para enriquecer a análise e facilitar a identificação de padrões relacionados à sobrevivência dos passageiros.

As seguintes features foram adicionadas:

- `tamanho_familia`: soma de `SibSp` e `Parch`, acrescida do próprio passageiro, representando o número total de pessoas da mesma família a bordo.
- `faixa_etaria`: agrupamento da variável `Age` em categorias de idade (Criança, Adolescente, Jovem, Adulto e Idoso).
- `valor_tarifa`: categorização da variável `Fare` com base em seus quartis, permitindo comparar passageiros por faixas de valor da tarifa paga.
- `situacao`: transformação da variável `Survived` em uma variável categórica para tornar a interpretação dos resultados mais intuitiva.
- `Sex`: seu nome e valores foram traduzidos para tornar a interpretação dos resultados mais intuitiva.

Essas novas variáveis permitem realizar análises mais interpretáveis e investigar como fatores como idade, tamanho da família e faixa de tarifa podem estar relacionados à sobrevivência dos passageiros.

---

# **6 - Análise Exploratória (EDA)**

# Sobrevivência Geral

## Objetivo

Avaliar a distribuição geral da sobrevivência ao naufrágio do Titanic. Essa análise servirá como referência para comparar a relação entre a sobrevivência e diferentes características dos passageiros nas próximas análises.

## Perguntas de negócio

- Qual foi a taxa geral de sobrevivência dos passageiros?
- Qual foi a quantidade absoluta de sobreviventes e não sobreviventes?

## Métricas analisadas

Para compreender o cenário geral de sobrevivência, serão avaliados:

- Quantidade absoluta de passageiros sobreviventes e não sobreviventes;
- Taxa de sobrevivência.

In [ ]:
# Quantidade absoluta de sobreviventes e não sobreviventes
qtde_geral = statistics.calcular_quantidade(df_limpo, "situacao")

# Taxa de Sobrevivência Geral
taxa_situacao = (
    df_limpo["situacao"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="percentual")
)

In [ ]:
display(qtde_geral)
display(taxa_situacao)

In [ ]:
grafico_barras(
    data=taxa_situacao,
    x="situacao",
    y="percentual",
    titulo="Taxa de Sobrevivência Geral",
    xlabel="Situação",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    nome_labels=["Não Sobreviveu", "Sobreviveu"],
    salvar_em="../reports/figures/sobrevivencia_geral.png"
)

## Principais resultados

Dos 891 passageiros analisados, 342 sobreviveram ao desastre, enquanto 549 não sobreviveram.

Mais da metade dos passageiros não sobreviveu ao naufrágio. Dessa forma, é possível analisar quais características podem estar associadas à sobrevivência dos passageiros.

Nas próximas análises serão avaliados fatores como:

- Gênero;
- Classe do passageiro;
- Faixa etária;
- Tamanho da família;
- Valor da tarifa.

# Sobrevivência por Gênero

## Objetivo

Analisar se o gênero tem relação com a sobrevivência dos passageiros e comparar se algum gênero teve maior taxa de sobrevivência.

## Perguntas de negócio

- Qual gênero apresentou maior taxa de sobrevivência?
- O maior número absoluto de sobreviventes é consistente com a maior taxa de sobrevivência?

## Métricas analisadas

- Total de passageiros por gênero;
- Total de sobreviventes por gênero;
- Taxa de sobrevivência por gênero;

In [ ]:
# Pega as estatísticas de quantas pessoas, a taxa de sobrevivência e a situação por gênero
estatisticas_genero = statistics.calcular_estatisticas_sobrevivencia(df_limpo, "sexo")

display(estatisticas_genero["quantidade"])
display(estatisticas_genero["taxa"])
display(estatisticas_genero["quantidade_situacao"])

In [ ]:
grafico_barras(
    data=estatisticas_genero["taxa"],
    x="sexo",
    y="taxa_sobrevivencia",
    titulo="Taxa de Sobrevivência por Gênero",
    xlabel="Gênero",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    salvar_em="../reports/figures/sobrevivencia_genero.png"
)

## Principais resultados

Dos 891 passageiros analisados, 577 eram homens e 314 eram mulheres. Essa diferença na composição da amostra deve ser considerada ao comparar as taxas de sobrevivência entre os grupos.

Mais da metade das mulheres sobreviveram, enquanto dos homens sobreviveram cerca de um quinto.

A diferença observada entre os grupos indica uma associação entre gênero e sobrevivência no desastre, com mulheres apresentando uma taxa de sobrevivência consideravelmente maior.

Apesar do número menor de mulheres, foi possível observar que o gênero feminino teve a maior taxa e também o  maior número absoluto de sobreviventes.

Esse resultado sugere investigar outros fatores que também podem estar relacionados à sobrevivência, como classe do passageiro, idade e tamanho da família.

# Sobrevivência por Classe

## Objetivo

Avaliar se a classe foi um fator relacionado à sobrevivência dos passageiros do Titanic, comparando a taxa de sobrevivência das classes.

## Perguntas de negócio

- Qual classe apresentou maior taxa de sobrevivência?
- Qual a diferença entre a taxa de sobrevivência das classes?
- As classes com maior número de sobreviventes também apresentaram as maiores taxas de sobrevivência?

## Métricas analisadas

- Total de passageiros por classe;
- Total de sobreviventes por classe;
- Taxa de sobrevivência por classe;

In [ ]:
# Pega as estatísticas de quantas pessoas, a taxa de sobrevivência e a situação por classe
estatisticas_classe = statistics.calcular_estatisticas_sobrevivencia(df_limpo, "Pclass")

display(estatisticas_classe["quantidade"])
display(estatisticas_classe["taxa"])
display(estatisticas_classe["quantidade_situacao"])

In [ ]:
grafico_barras(
    data=estatisticas_classe["taxa"],
    x="Pclass",
    y="taxa_sobrevivencia",
    titulo="Taxa de Sobrevivência por Classe",
    xlabel="Classe do Passageiro",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    nome_labels=["1ª Classe", "2ª Classe","3ª Classe"],
    salvar_em="../reports/figures/sobrevivencia_classe.png"
)

## Principais resultados

Dos 891 passageiros analisados, 216 eram da 1ª classe, 184 da 2ª classe e 491 da 3ª classe. Essa diferença na composição da amostra deve ser considerada ao comparar as taxas de sobrevivência entre os grupos.

Mais da metade dos passageiros da 1ª classe sobreviveram, enquanto da 2ª classe aproximadamente metade e da 3ª classe cerca de um quarto.

A diferença observada entre os grupos indica uma associação entre classe e sobrevivência no desastre, demonstrando uma possível tendência de quanto maior o nível da classe maior a taxa de sobrevivência.

Também é possível dizer que apesar da menor quantidade de passageiros, tiveram mais sobreviventes da 1ª classe do que da 3ª classe, com isso o mesmo grupo que possui a maior taxa de sobrevivência é também o maior em valor absoluto. Já a 2ª classe teve menos sobreviventes que a 3ª classe, porém teve uma taxa de sobrevivência maior.

Esse resultado sugere investigar outros fatores que também podem estar relacionados à sobrevivência, como a idade e tamanho da família, além de cruzar variáveis como classe e gênero.

# Sobrevivência por Classe e Gênero

## Objetivo

Cruzar os dados entre classe e gênero e analisar como se comporta a sobrevivência entre os grupos formados pelos gêneros e classes.

## Perguntas de negócio

- Qual grupo teve a maior taxa de sobrevivência?
- Qual a diferença entre a taxa de sobrevivência dos grupos?

## Métricas analisadas

- Total de passageiros por grupo;
- Total de sobreviventes por grupo;
- Taxa de sobrevivência por grupo;

In [ ]:
# Taxa de Sobrevivência por classe e gênero
estatistica_classe_genero = statistics.calcular_estatisticas_sobrevivencia_grupo(df_limpo, "sexo", "Pclass")

display(estatistica_classe_genero["quantidade"])
display(estatistica_classe_genero["taxa"])
display(estatistica_classe_genero["quantidade_situacao"])

In [ ]:
grafico_barras(
    data=estatistica_classe_genero["taxa"],
    x="Pclass",
    y="taxa_sobrevivencia",
    hue="sexo",
    titulo="Taxa de Sobrevivência por Classe e Sexo",
    xlabel="Classe do Passageiro",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    nome_labels=["1ª Classe", "2ª Classe","3ª Classe"],
    mostrar_legenda=True,
    salvar_em="../reports/figures/sobrevivencia_classe_genero.png"
)

## Principais resultados

Dos 891 passageiros analisados, 216 eram da 1ª classe, sendo 122 homens e 94 mulheres, 184 da 2ª classe, sendo 108 homens e 76 mulheres e 491 da 3ª classe, sendo 347 homens e 144 mulheres. Essa diferença na composição da amostra deve ser considerada ao comparar as taxas de sobrevivência entre os grupos.

Dentre as mulheres da 1ª e 2ª classe sobreviveram quase todas, já na 3ª classe sobreviveram metade delas. Já nos homens da 1ª classe sobreviveram menos da metade e na 2ª e 3ª não chegou nem a um quinto de sobreviventes.

É possível perceber que mesmo dentro da própria classe as mulheres continuam apresentando uma taxa de sobrevivência maior que a dos homens, enquanto a queda de sobrevivência é mais acentuada na 3ª classe.

Mesmo com a tendência de quanto maior o nível da classe maior a taxa de sobrevivência, mulheres da 3ª classe possuem taxa de sobrevivência maior que homens da 1ª classe.

# Sobrevivência por Faixa Etária

## Objetivo

Avaliar se a idade foi um fator relacionado à sobrevivência dos passageiros do Titanic, comparando a taxa de sobrevivência das faixas etárias.

## Perguntas de negócio

- Existe uma tendência na taxa de sobrevivência conforme a faixa etária aumenta?
- Essa tendência é contínua ou há exceções?

## Métricas analisadas

- Total de passageiros por faixa etária;
- Total de sobreviventes por faixa etária;
- Taxa de sobrevivência por faixa etária;

### Faixas etárias utilizadas na análise

Para facilitar a interpretação dos resultados, a idade foi agrupada nas seguintes categorias. Os intervalos foram definidos de forma a reduzir a concentração de observações em uma única categoria, permitindo comparações mais equilibradas entre as faixas etárias.

- Criança: 0–12 anos
- Adolescente: 13–17 anos
- Jovem: 18–24 anos
- Adulto: 25–59 anos
- Idoso: 60 anos ou mais

In [ ]:
# Pega as estatísticas de quantas pessoas, a taxa de sobrevivência e a situação por faixa etária
estatisticas_faixa_etaria = statistics.calcular_estatisticas_sobrevivencia(df_limpo, "faixa_etaria")

display(estatisticas_faixa_etaria["quantidade"])
display(estatisticas_faixa_etaria["taxa"])
display(estatisticas_faixa_etaria["quantidade_situacao"])

In [ ]:
grafico_barras(
    data=estatisticas_faixa_etaria["taxa"],
    x="faixa_etaria",
    y="taxa_sobrevivencia",
    titulo="Taxa de Sobrevivência por Faixa Etária",
    xlabel="Faixa Etária",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    nome_labels=["Criança", "Adolescente", "Jovem", "Adulto", "Idoso"],
    salvar_em="../reports/figures/sobrevivencia_faixa_etaria.png"
)

## Principais resultados

Dos 891 passageiros analisados, 546 são adultos, 206 jovens, 69 crianças, 44 adolescentes e 26 idosos. Essa diferença na composição da amostra deve ser considerada ao comparar as taxas de sobrevivência entre os grupos.

Dentre as faixas é possível identificar uma tendência de quanto maior a idade menor a taxa de sobrevivência, também observa-se que a tendência é contínua, não tendo nenhum grupo como exceção.

É importante pontuar que a interpretação da taxa dos idosos deve ser feita com cautela devido ao baixo número de observações.

# Sobrevivência por Tamanho da Família

## Objetivo

Avaliar se o tamanho da família foi um fator relacionado à sobrevivência dos passageiros do Titanic, comparando a taxa de sobrevivência dos diferentes tamanhos de família.

## Perguntas de negócio

- Famílias com mais pessoas tiveram maior taxa de sobrevivência?
- Pessoas sozinhas tiveram maiores chances de sobreviver?
- Existe um tamanho de família que apresenta maior taxa de sobrevivência?

## Métricas analisadas

- Taxa de sobrevivência por tamanho da família;
- Distribuição dos passageiros por tamanho da família;

In [ ]:
# Pega as estatísticas de quantas famílias, a taxa de sobrevivência.
estatisticas_tamanho_familia = statistics.calcular_estatisticas_sobrevivencia(df_limpo, "tamanho_familia")

display(estatisticas_tamanho_familia["quantidade"])
display(estatisticas_tamanho_familia["taxa"])

In [ ]:
grafico_barras(
    data=estatisticas_tamanho_familia["taxa"],
    x="tamanho_familia",
    y="taxa_sobrevivencia",
    titulo="Taxa de Sobrevivência por Tamanho da Família",
    xlabel="Tamanho da Família",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    salvar_em="../reports/figures/sobrevivencia_tamanho_familia.png"
)

## Principais resultados

Dos 891 passageiros, 537 estavam sozinhos, 161 em duas pessoas, 102 em três pessoas. Os demais grupos possuem menos de 30 passageiros, o que deve ser considerado na interpretação das taxas.

Famílias de duas a quatro pessoas tiveram mais da metade de taxa de sobrevivência, já com famílias maiores do que quatro pessoas a taxa cai bastante em alguns casos ficando com nenhum sobrevivente.

Já pessoas que viajaram sozinhas tiveram cerca de um terço de taxa de sobrevivência.

Isso indica que viajar sozinho não aumentava a chance de sobreviver, que famílias com tamanho intermediário de duas a quatro pessoas tinham maiores chances de sobrevivência e que apesar da baixa taxa das famílias muito grandes é preciso ter cautela ao interpretar esses valores devido ao baixo número de observações.

# Sobrevivência por Valor da Tarifa

## Objetivo

Avaliar se o valor da tarifa foi um fator relacionado à sobrevivência dos passageiros do Titanic, comparando a taxa de sobrevivência dos diferentes quartis da tarifa.

## Perguntas de negócio

- Existe tendência de quanto maior a faixa de valor maior a taxa de sobrevivência?
- Essa tendência ocorre em todas as faixas ou há exceções?
- A diferença entre as faixas é expressiva?

## Métricas analisadas

- Taxa de sobrevivência por quartis do valor da tarifa

### Faixas de valor da tarifa utilizadas na análise

Para facilitar a interpretação dos resultados, o valor da tarifa foi agrupado nas seguintes categorias. Os intervalos foram determinados automaticamente pelos quartis para dividir os passageiros em grupos com quantidades semelhantes.

- Baixo: até £7,91
- Médio: £7,92 a £14,45
- Alto: £14,46 a £31,00
- Muito Alto: acima de £31,00

In [ ]:
# Taxa de sobrevivência por valor da tarifa
taxa_tarifa = statistics.calcular_taxa_sobrevivencia(df_limpo, "valor_tarifa")
taxa_tarifa

In [ ]:
grafico_barras(
    data=taxa_tarifa,
    x="valor_tarifa",
    y="taxa_sobrevivencia",
    titulo="Taxa de Sobrevivência por Valor da Tarifa",
    xlabel="Classificação dos Valores",
    ylabel="Taxa de Sobrevivência (%)",
    percentual=True,
    salvar_em="../reports/figures/sobrevivencia_valor_tarifa.png"
)

## Principais resultados

Os 891 passageiros foram divididos em 4 quartis, nos valores de Baixo, Médio, Alto e Muito Alto. Dessa forma os quartis produziram grupos com quantidades semelhantes de passageiros.

É possível perceber que existe uma tendência de quanto maior o valor pago na tarifa maior a taxa de sobrevivência, ocorrendo de forma contínua em todas as faixas.

Também existe uma diferença considerável entre as faixas, com a faixa Baixa não chegando a um quinto de taxa de sobrevivência, enquanto a faixa Muito Alto tem taxa acima da metade.

O comportamento observado é consistente com a análise por classe, já que passageiros que pagaram tarifas mais altas também apresentaram maiores taxas de sobrevivência.

---

# **7 - Conclusão**

## Objetivo da análise

Identificar quais os fatores estão associados à sobrevivência no Titanic, assim permitindo compreender o perfil dos sobreviventes.. 

## Principais achados

Mulheres apresentam maior taxa de sobrevivência que os homens;
Mulheres da terceira classe apresentam mais chance de sobrevivência do que homens da primeira;
Passageiros da primeira classe apresentam maior taxa de sobrevivência que as outras classes;
Dentre as faixas etárias as crianças apresentam a maior taxa de sobrevivência;
Famílias de 2 a 4 pessoas apresentam maior chance de sobrevivência;
Valores de tarifas mais altos apresentam mais chance de sobrevivência.

## Limitações

O estudo identifica associações, não causas. Além da limitação pelo número de amostras nos grupos dos idosos e das famílias grandes.

## Considerações finais

Dentre as variáveis as que parecem mais estar associadas a sobrevivência são gênero, classe e valor da tarifa.

Sendo o perfil de passageiros com maiores chances de sobrevivência:

- Mulher;
- Primeira Classe;
- Criança;
- Em famílias com tamanho de 2 a 4 pessoas.

Os resultados obtidos fornecem uma visão geral dos fatores associados à sobrevivência e podem servir como base para análises mais aprofundadas, como modelos preditivos ou estudos multivariados.
 